# OpenSearch ESCI検索改善 実験サマリ

このnotebookは、技術書典向け原稿を書くための参考資料である。本文そのものではなく、実験の流れ、主要な数値、実装上の判断、後から参照すべきnotebook/scriptを整理する。

題材はAmazon ESCI日本語データをOpenSearchへ投入し、lexical search、LTR/GBDT、semantic hybrid searchで検索品質を改善する一連の検証である。

## 0. 書籍の趣旨と読者に伝えたいこと

### 位置付け

- OpenSearch利用の応用編。
- 単なるAPI利用ではなく、検索品質改善の実験プロセスを扱う。
- embedding、k-NN、LTR plugin、XGBoost/GBDT、ランキング指標を実データでつなげて説明する。
- 技術書典の同人誌として、手元環境で試せる実験ログと、理論補足の両方を狙う。

### 読者に持ち帰ってほしいこと

- 検索改善では、まず問題分析と評価指標を用意する必要がある。
- OpenSearchの標準検索、LTR plugin、k-NN searchは組み合わせられるが、実装順序やscoreの扱いに制約がある。
- GBDTは特徴量設計と評価設計が肝で、OpenSearch LTR pluginではfeature loggingとrescoreが中心になる。
- semantic searchは最終ランキング特徴量に入れられなくても、候補生成のrecall改善として使える。

## 1. 実験全体の流れ

| Stage | 施策 | 主な狙い | 主な成果 |
|---|---|---|---|
| v0 | データ投入・初期index | ESCI JP商品をOpenSearchへ投入 | 339,059商品を投入 |
| v1 | baseline lexical | 初期検索品質と問題を把握 | 問題分析・評価指標の必要性を確認 |
| v2 | lexical tuning | 日本語検索向けにfield/analyzer/boostを調整 | candidate評価・end-to-end評価が改善 |
| v3 | XGBoost LTR | lexical/構造化特徴量をGBDTで統合 | test official nDCG 0.84947→0.85548 |
| v4 | semantic hybrid | k-NNで候補生成recallを改善 | test LTR nDCG@10 0.42539→0.42781 |

本文ではこの順番で書くと自然。いきなり高度なモデルへ飛ばず、問題分析・評価・改善の順に積み上げる。

## 2. 実験環境・データセット

### OpenSearch

- ローカルDocker上のOpenSearchを利用。
- 主要plugin:
  - `opensearch-ltr`
  - `opensearch-knn`
  - `opensearch-neural-search`
  - 日本語解析用にKuromoji系analysisを利用。

### データセット

- Amazon ESCI Shopping Queries Dataset。
- `product_locale == 'jp'` を対象。
- 商品数: 339,059。
- 評価は主に `small_version == 1` の日本語queryで実施。
- train内validationは `SHA1(query_id) % 5 == 0` でquery単位に固定分割。

### 主要index

| Index | 内容 |
|---|---|
| `amazon-jp-v2` | 日本語lexical/LTR用の本体index |
| `amazon-jp-semantic-v1` | `amazon-jp-v2` 相当のfield + `product_embedding` を持つsemantic index |

### 使用モデル・ライブラリ

- XGBoost `XGBRanker`
- OpenSearch LTR plugin XGBoost JSON model
- `intfloat/multilingual-e5-base`
- PyTorch MPS/Metalをembedding生成に利用

### 手元実行環境メモ

- M5 MacBook Air / 16GB RAM。
- MPSは利用可能だったが、長文embeddingでは失速する場面があった。
- 最終的な全商品embeddingは `max_seq_length=64`, `bullet_chars=0` で作成。

## 3. データ投入とindex設計

### データ投入で重要だったこと

- JP商品339,059件が全てfeedされているかを最初に確認する。
- pandas上の `NaN` をJSONへそのまま流すとbulk ingestやmappingで問題になりやすい。
- `where(pd.notna(df_products), None)` のように、欠損値をOpenSearchへ渡せる `None/null` に変換する処理が重要。

### 本文で書くと良いポイント

- 検索改善以前に「データが正しく入っているか」が最重要。
- 途中で日本語ドキュメントが一部しかfeedされない問題を見つけ、全件投入を確認した。
- count APIで `339,059` 件を確認する。

In [ ]:
# データ投入確認の例
# curl -sS http://localhost:9200/amazon-jp-v2/_count
# curl -sS http://localhost:9200/amazon-jp-semantic-v1/_count

# pandasでNaNをNoneに寄せるイメージ
# df_products = df_products.where(pd.notna(df_products), None)

## 4. 問題分析

### 目的

- 初期検索でどのようなqueryが苦手かを把握する。
- 改善施策を思いつきではなく、失敗パターンから導く。
- 後続のlexical特徴量、GBDT特徴量、semantic searchの必要性を説明する材料にする。

### 対象query

- `shopping_queries_dataset_examples` から日本語queryをサンプリング。
- 最初は10件、その後30件へ拡張して問題分析。
- query typeの例:
  - ブランド + カテゴリ
  - 型番・数値条件
  - 日本語複合語
  - 用途・課題解決系
  - 否定条件
  - 作品名・entity

### 観察された問題

- OR寄りのBM25では、重要語の一部だけ一致した商品が上がる。
- 型番・数字・ブランド表記揺れに弱い。
- 否定条件や「なし」「非対応」の扱いが難しい。
- 日本語の読み・表記揺れ・複合語に対するanalyzer設計が重要。
- semanticには拾えるがlexicalでは拾いにくい商品がある一方、semanticは条件違いのノイズも拾う。

### 参照ファイル

- `amazon/esci_improvements.ipynb`
- `amazon/artifacts/ltr_xgboost_v3/diagnostic_30.csv`
- `amazon/artifacts/ltr_xgboost_v3/diagnostic_30_summary.json`

## 5. 評価指標と評価設計

### なぜ評価指標が必要か

検索改善は目視だけでは判断できない。特にランキングでは「正しい商品が含まれているか」と「上位に来ているか」が別の問題になる。そのため、改善施策の前に評価指標を導入する。

### ESCI labelとgain

| Label | 意味 | gain |
|---|---|---:|
| E | Exact | 1.0 |
| C | Complement | 0.1 |
| S | Substitute | 0.01 |
| I | Irrelevant | 0.0 |

### nDCGの基本

- DCG: 上位ほど大きく割り引いてgainを足す。
- IDCG: 理想的な順位でのDCG。
- nDCG: `DCG / IDCG`。
- queryごとにnDCGを計算し、query平均を取る。

### 今回使った評価の種類

| 評価 | 何を見るか | 用途 |
|---|---|---|
| 公式Task 1風 candidate reranking | ESCIの判定済み候補内での並べ替え | モデル単体・特徴量追加の効果を見る |
| OpenSearch end-to-end | 実際にOpenSearchへqueryを投げる | 候補生成 + rerank全体を見る |
| nDCG@10 unjudged=0 | 全商品検索runで未判定を0扱い | 上位表示品質の参考値 |
| Recall@100 judged positive | LTR前候補に正例が入ったか | semantic候補生成の効果を見る |
| judged@10 | 上位10件のうち判定済み商品の割合 | 未判定が多い評価の信頼度を見る補助指標 |

### 未判定商品の扱い

- 全商品検索ではESCIのqrelsにない商品が多数返る。
- TREC風には未判定をgain 0として扱うことが多い。
- ただしESCI qrelsは全商品検索の完全poolではないため、`judged@10` を併記し、過度に主指標化しない。

In [ ]:
# nDCG実装の最小例
import numpy as np

GAINS = {"I": 0.0, "S": 0.01, "C": 0.1, "E": 1.0}

def ndcg(labels, k=None):
    gains = np.array([GAINS[x] for x in labels], dtype=float)
    if k is not None:
        gains = gains[:k]
    discounts = np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(gains / discounts)
    ideal = np.sort(gains)[::-1]
    idcg = np.sum(ideal / discounts)
    return float(dcg / idcg) if idcg else 0.0

## 6. 改善1: lexical tuning

### What

- 日本語向けindex `amazon-jp-v2` を作成。
- Kuromoji、readingform、2–3gram、raw fieldなどを利用。
- title/brand/bullet/descriptionのboostを調整。
- query単位のtrain内validationでboostを選択。

### Why

- 初期BM25では日本語表記揺れ、読み、複合語に弱い。
- EC検索ではtitle/brandの一致が特に重要。
- bullet/descriptionはrecallには効くが、強すぎるとノイズになる。

### How

- `small_version=1, split=train, locale=jp` をquery単位でtrain/validation分割。
- validationの公式nDCGでboost候補を比較。
- testは最後の確認用に使う。

### 結果の読み方

- lexical tuningは大幅な順位改善というより、zero-match減少やcandidate coverage改善に効く。
- 後続のLTR/GBDTに渡す土台として重要。

### 参照ファイル

- `amazon/esci_lexical_v2.ipynb`
- `amazon/esci_eval.ipynb`
- `amazon/ltr_features.py` の `feature_set` / `lexical_v2`

In [ ]:
# lexical_v2 queryの雰囲気。実体は amazon/ltr_features.py を参照。
lexical_v2 = {
    "bool": {
        "minimum_should_match": 1,
        "should": [
            {"match_phrase": {"product_title": {"query": "{{keywords}}", "boost": 8.0}}},
            {"match": {"product_title": {"query": "{{keywords}}", "boost": 4.0, "operator": "or"}}},
            {"match_phrase": {"product_brand": {"query": "{{keywords}}", "boost": 6.0}}},
            {"match": {"product_brand": {"query": "{{keywords}}", "boost": 3.0, "operator": "or"}}},
            {"match": {"product_bullet_point": {"query": "{{keywords}}", "boost": 0.5}}},
            {"match": {"product_description": {"query": "{{keywords}}", "boost": 0.1}}},
        ],
    }
}

## 7. 改善2: XGBoost LTR / GBDT導入

### What

- OpenSearch LTR pluginでfeature setを定義。
- OpenSearchのfeature loggingでquery-product特徴量を収集。
- XGBoost `XGBRanker` でLTRモデルを学習。
- XGBoost JSONをOpenSearch LTR pluginへdeployし、`rescore` でrerank。

### Why

- 単一BM25 scoreでは、brand一致、phrase一致、型番一致、否定条件などをうまく統合できない。
- GBDTなら複数特徴量の非線形な相互作用を学習できる。
- LTR pluginを使うことで、学習時とOpenSearch検索時の特徴量計算を揃えられる。

### v2→v3で追加した特徴量

- title/brand/bullet/descriptionのAND一致。
- raw exact一致。
- color一致。
- 数字・型番token。
- 否定語・反対条件。
- query token count / numeric / latin / negation / short queryなどのquery側特徴。

### 主要結果

| 評価 | v2 | v3 | 差分 |
|---|---:|---:|---:|
| validation official nDCG | 0.844872 | 0.851465 | +0.006593 |
| test official nDCG | 0.849470 | 0.855483 | +0.006013 |
| corpus nDCG@10 unjudged=0 | 0.402380 | 0.424668 | +0.022288 |
| corpus nDCG@100 unjudged=0 | 0.475101 | 0.487602 | +0.012502 |

validationのpaired bootstrap 95% CIは `[+0.003214, +0.009986]`。testでも改善を確認。

### 実装上の注意

- OpenSearch LTRのfeatureは検索queryとして定義する。
- feature queryがmatchしないとNaNになりやすいため、tinyなmatch_all scoreでdense化した。
- LTR pluginのXGBoost JSON evaluatorではleaf scoreが負だと扱いづらいため、leafを非負にshiftし、offsetで順位同一性を保った。
- `query_weight=0.0`, `rescore_query_weight=1.0` とし、最終順位をLTR scoreで決めた。

### 参照ファイル

- `amazon/esci_ltr_xgboost.ipynb`
- `amazon/esci_ltr_xgboost_v3.ipynb`
- `amazon/run_ltr_xgboost.py`
- `amazon/run_ltr_xgboost_v3.py`
- `amazon/ltr_features.py`
- `amazon/artifacts/ltr_xgboost_v3/metrics.json`

In [ ]:
# LTR feature loggingの最小イメージ
body = {
    "size": 100,
    "query": {
        "bool": {
            "must": [{"ids": {"values": ["PRODUCT_ID_1", "PRODUCT_ID_2"]}}],
            "filter": [{
                "sltr": {
                    "_name": "logged_features",
                    "featureset": "esci_jp_features_v3",
                    "params": {"keywords": "自転車 スピーカー"},
                }
            }],
        }
    },
    "ext": {
        "ltr_log": {
            "log_specs": {"name": "features", "named_query": "logged_features"}
        }
    },
}

## 8. 理論補足メモ: GBDT / LambdaRank / LTR

本文で深掘りするとよい話題。

### Learning to Rankの種類

- Pointwise: 各query-productを独立にスコア回帰/分類。
- Pairwise: 同じquery内の商品ペアの順序を学習。
- Listwise: nDCGなどランキング全体の指標を意識して学習。

### XGBoost Ranker

- `rank:pairwise` と `rank:ndcg` を比較した。
- groupにはqueryごとの候補数を渡す。
- 今回のv3 selected configは `rank:ndcg`, depth 5, 150 trees。

### GBDTが検索で使いやすい理由

- featureのスケール差に比較的強い。
- 欠損や非線形な条件分岐を扱いやすい。
- ルールベースで作った検索特徴量を自然に統合できる。

### OpenSearch LTR pluginとの関係

- OpenSearch側で特徴量をqueryとして計算する。
- 学習はPythonで行い、モデルをOpenSearchにdeployする。
- 最終検索では`rescore` window内だけをrerankするため、前段候補生成が重要。

## 9. 改善3: semantic hybrid search

### What

- `intfloat/multilingual-e5-base` で商品embeddingを作成。
- `amazon-jp-semantic-v1` に `knn_vector` fieldを追加。
- `bool.should` でlexical queryとk-NN queryを同時に投げる。
- そのtop100をOpenSearch LTR pluginのv3 XGBoostでrescoreする。

### Why

- lexicalだけでは語彙差が大きい商品を拾えない。
- semantic searchは候補生成recallを改善できる。
- 一方でsemantic scoreをXGBoost特徴量に入れるのはLTR plugin上で難しかったため、候補生成だけに使った。

### 採用した構成

```text
bool.should
  ├─ lexical_v2 query
  └─ k-NN query with semantic_boost β
↓
OpenSearch raw _scoreでtop100候補選定
↓
LTR plugin esci_jp_xgb_v3でrescore
```

### Embedding条件

- product: `passage: title | brand | color`
- query: `query: ...`
- `max_seq_length=64`
- `bullet_chars=0`
- 商品embedding生成: 339,059件 / 約33.7分
- bulk ingest: 約7分

### 主要結果

Validation smallでは `semantic_boost=0.3` がLTR後nDCG@10 best。

| boost | LTR nDCG@10 | LTR nDCG@100 | recall@100 |
|---:|---:|---:|---:|
| 0.0 | 0.428007 | 0.484222 | 0.472151 |
| 0.3 | 0.429595 | 0.488345 | 0.480253 |
| 1.0 | 0.429387 | 0.488869 | 0.481894 |

Test smallでは `semantic_boost=0.5` がLTR後nDCG@10 best。

| boost | LTR nDCG@10 | LTR nDCG@100 | recall@100 |
|---:|---:|---:|---:|
| 0.0 | 0.425388 | 0.488270 | 0.485383 |
| 0.3 | 0.427698 | 0.492839 | 0.492309 |
| 0.5 | 0.427806 | 0.493316 | 0.493210 |
| 1.0 | 0.427231 | 0.493497 | 0.494121 |

Test `boost=0.5` のbaseline比:

- nDCG@10: +0.002417, 95% CI `[+0.001313, +0.003671]`
- nDCG@100: +0.005046, 95% CI `[+0.003733, +0.006518]`
- recall@100: +0.007827, 95% CI `[+0.006042, +0.009807]`

### 考察

- semantic k-NNはrecall改善に効いた。
- LTR v3は追加候補をある程度うまく並べ替えられた。
- boostを上げるほどrecall@100は増えるが、nDCG@10は0.5付近で頭打ち。
- 採用候補は保守的には0.3、効果重視なら0.5。

### 参照ファイル

- `amazon/semantic_embeddings.py`
- `amazon/run_hybrid_bool_ltr_v4.py`
- `amazon/esci_hybrid_bool_ltr_v4.ipynb`
- `amazon/artifacts/hybrid_bool_ltr_v4/metrics_summary_test_small.json`

In [ ]:
# bool.should lexical+kNN + LTR rescoreの構成イメージ
body = {
    "size": 100,
    "query": {
        "bool": {
            "should": [
                {"... lexical_v2 query ...": {}},
                {
                    "knn": {
                        "product_embedding": {
                            "vector": [0.0, 0.1, 0.2],  # 実際は768次元
                            "k": 100,
                            "boost": 0.5,
                        }
                    }
                },
            ],
            "minimum_should_match": 1,
        }
    },
    "rescore": {
        "window_size": 100,
        "query": {
            "rescore_query": {
                "sltr": {
                    "model": "esci_jp_xgb_v3",
                    "params": {"keywords": "自転車 スピーカー"},
                }
            },
            "query_weight": 0.0,
            "rescore_query_weight": 1.0,
        },
    },
}

## 10. 理論補足メモ: embedding / HNSW / hybrid search

### embedding検索

- queryとdocumentを同じベクトル空間へ写像する。
- E5系モデルではprefixが重要: queryは `query: ...`、documentは `passage: ...`。
- cosine similarityまたは正規化済みベクトルの内積で近さを測る。

### HNSW

- approximate nearest neighbor searchの代表的手法。
- 階層化された近傍グラフをたどって近いベクトルを高速に探す。
- 主なパラメータ:
  - `m`: 各ノードが持つ辺数の目安。
  - `ef_construction`: index構築時の探索幅。
  - 検索時の探索幅は精度・速度トレードオフに関わる。

### hybrid searchのscore問題

- BM25 scoreとk-NN scoreはスケールが違う。
- OpenSearchにはnormalization processorやRRFがある。
- ただし、normalization後topNをLTR rescoreする順序を1リクエストで作るのは難しかった。
- 今回は単純な `bool.should` とboost tuningで対応した。

### 本文での説明ポイント

- semantic searchは万能ではなく、条件違いの商品も拾う。
- EC検索ではsemantic recallとlexical exactnessのバランスが重要。
- LTRがある場合、semanticは候補生成に使うだけでも効果がある。

## 11. 採用しなかった案・詰まった点

成功ルートだけでなく、ここは本文の読みどころになりそう。

### 1. semantic scoreをLTR pluginのXGBoost特徴量に入れる案

- OpenSearch k-NN script scoring自体は、literalなquery vector配列なら動く。
- しかしLTR feature template経由で `query_vector` を渡すと、Mustache展開で文字列化される。
- `knn_score` が期待する数値配列として受け取れず、現環境では実用困難だった。
- 結果として、semantic scoreをXGBoost特徴量に入れる案は見送った。

### 2. search pipeline normalization processor案

- lexical scoreとsemantic scoreをmin-max正規化して線形結合する案を検討した。
- ただしOpenSearch標準のhybrid query + normalization + rescoreでは、normalization後topNをLTRでrescoreする順序を1リクエスト内で作りにくい。
- 2度引きなら可能だが、今回はLTR pluginに閉じた1リクエスト方式を優先した。

### 3. 外部reranker案

- search pipelineから外部GBDT serviceを呼ぶ案も調査した。
- semantic特徴量込みGBDTをやるなら本命に近い。
- ただし今回はOpenSearch LTR pluginに閉じる方針だったため採用しなかった。

### 4. embedding生成の計算コスト

- MPSは使えたが、長文bullet込みだと失速した。
- `max_seq_length=64`, `bullet_chars=0` にして実験を前に進めた。
- この判断は速度優先。精度検証としては、後でbullet短め版と比較する余地がある。

## 12. 計算コスト・運用上のメモ

### embedding生成

| 条件 | メモ |
|---|---|
| model | `intfloat/multilingual-e5-base` |
| device | MPS/Metal |
| final product text | title + brand + color |
| max_seq_length | 64 |
| product count | 339,059 |
| embedding生成時間 | 約33.7分 |
| `.npy` size | 約993MB |
| bulk ingest | 約7分 |

### MPS/CPU比較で得た教訓

- 10240件ベンチではMPSがCPUより速かった。
- ただし長時間・長文ではMPSが失速し、tokenizationもボトルネックになる。
- batch sizeは大きければ良いわけではなく、16〜32付近が良かった。

### OpenSearch運用で後から見る点

- k-NN indexのメモリ使用量。
- HNSWパラメータ。
- `rescore.window_size` とレイテンシ。
- semantic boostとレイテンシ/品質のトレードオフ。
- refresh/replica/mergeの設定。

## 13. 最終結果まとめ

### 段階別の代表値

| Stage | 指標 | 値 |
|---|---|---:|
| v2 lexical/LTR初期 | test XGBoost official nDCG | 0.849315 |
| v3 XGBoost LTR | test official nDCG | 0.855483 |
| v3 XGBoost LTR | corpus nDCG@10 unjudged=0 | 0.424668 |
| v4 semantic hybrid + LTR | test nDCG@10 unjudged=0 | 0.427806 |
| v4 semantic hybrid + LTR | test recall@100 judged positive | 0.493210 |

### 改善の方向性

- lexical tuning: 日本語検索の土台を改善。
- XGBoost LTR: 候補集合が同じでも順位を改善。
- semantic hybrid: LTRへ渡る候補集合を改善し、recallとnDCGをさらに改善。

### 本文で強調したい結論

OpenSearchの検索改善は、単一の強いモデルを入れる話ではなく、index設計、評価、特徴量、候補生成、rerankを段階的に積み上げる作業だった。LTR pluginに閉じる場合でも、semantic searchを候補生成に使うだけで改善は得られる。

## 14. 参照notebook / script一覧

### Notebook

| File | 内容 |
|---|---|
| `amazon/esci_tutorial.ipynb` | 初期データ投入・OpenSearch接続 |
| `amazon/esci_improvements.ipynb` | 初期問題分析 |
| `amazon/esci_eval.ipynb` | ESCI公式作法に寄せた評価 |
| `amazon/esci_lexical_v2.ipynb` | lexical v2 tuning |
| `amazon/esci_ltr_xgboost.ipynb` | 初期XGBoost LTR |
| `amazon/esci_ltr_xgboost_v3.ipynb` | v3特徴量・XGBoost LTR本命 |
| `amazon/esci_semantic_hybrid_v4.ipynb` | semantic特徴量込み外部reranker検討 |
| `amazon/esci_hybrid_bool_ltr_v4.ipynb` | 採用したsemantic hybrid + LTR検証 |
| `amazon/summary.ipynb` | このサマリ |

### Python scripts

| File | 内容 |
|---|---|
| `amazon/ltr_features.py` | LTR feature set定義、XGBoost JSON export helper |
| `amazon/run_ltr_xgboost.py` | v2 LTR学習・評価 |
| `amazon/run_ltr_xgboost_v3.py` | v3 LTR学習・評価・deploy |
| `amazon/semantic_embeddings.py` | embedding生成、semantic index作成、bulk ingest |
| `amazon/run_semantic_hybrid_v4.py` | semantic特徴量込み外部reranker検討 |
| `amazon/run_hybrid_bool_ltr_v4.py` | 採用版: bool.should lexical+kNN + LTR評価 |

### Artifacts

| Directory | 内容 |
|---|---|
| `amazon/artifacts/ltr_xgboost/` | 初期XGBoost LTR結果 |
| `amazon/artifacts/ltr_xgboost_v3/` | v3 LTR結果 |
| `amazon/artifacts/semantic_hybrid/` | product embedding cache |
| `amazon/artifacts/hybrid_bool_ltr_v4/` | semantic hybrid + LTR結果 |

## 15. 本文執筆時のTODOメモ

- 問題分析queryを数件、具体例として本文に載せる。
- `diagnostic_30.csv` から改善/悪化が分かりやすいqueryを選ぶ。
- `example_improvements.csv` / `example_regressions.csv` からsemantic hybridの代表例を選ぶ。
- nDCGの数式を本文用に整える。
- HNSWの図を描くか検討する。
- LTR feature loggingの図を描くか検討する。
- OpenSearchのquery/rescore/search pipelineの実行順序を図示する。
- 失敗談として、semantic scoreをLTR特徴量に入れようとして詰まった話を入れる。
- 最終章では「手元Mac + OpenSearchでここまでできる」という締めにすると良さそう。